# DAKGEA - Esperimento Semplice

In questo notebook eseguiremo un esperimento baseline semplice:
- Dataset: D_W_15K_V1 (DBpedia-Wikidata)
- Reduction: 30% delle entità
- No augmentation (solo baseline)
- Model: BERT-INT

## Setup

In [ ]:
import sys
from pathlib import Path
import yaml

# Setup path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Creare la Configurazione

Una configurazione DAKGEA definisce:
- **name**: Nome dell'esperimento
- **dataset**: Quale dataset usare
- **reduction**: Come ridurre i dati
- **model**: Quale modello EA usare
- **seed**: Per riproducibilità

In [ ]:
# Definisci la configurazione
config = {
    "experiment": {
        "name": "tutorial_baseline",
        
        # Dataset configuration
        "dataset": {
            "name": "openea/D_W_15K_V1",  # DBpedia-Wikidata 15K entities
            "writer": "bert_int"          # Come scrivere i dati processati
        },
        
        # Reduction: usa solo 30% delle entità
        "reduction": {
            "method": "random_entities",   # Selezione random
            "ratio": 0.3,                  # 30% del dataset
            "writer": "bert_int",
            "eval": True,                  # Valuta il baseline
            "save_dataset": False,         # Non salvare il dataset ridotto
            "save_model": False            # Non salvare i modelli
        },
        
        # Model configuration
        "model": "bert_int",               # Usa BERT-INT
        "seed": 42,                        # Random seed
        "clear": True,                     # Pulisci cache intermedie
        "overwrite_existing": False        # Non sovrascrivere se esiste
    }
}

# Visualizza la config
print(yaml.dump(config, default_flow_style=False))

## 2. Salvare la Configurazione

In [ ]:
# Salva la configurazione in un file
config_file = PROJECT_ROOT / "config" / "experiments" / "tutorial_baseline.yaml"

with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to: {config_file}")

## 3. Eseguire l'Esperimento

### Opzione A: Da linea di comando

```bash
# Nel terminale
bash scripts/run_experiment.sh config/experiments/tutorial_baseline.yaml
```

### Opzione B: Da Python (in questo notebook)

In [ ]:
from experiments.runner.runner import ExperimentRunner
from src.config.loader import load_yaml

# Carica la configurazione
config_data = load_yaml(config_file)

# Crea il runner
runner = ExperimentRunner(
    payload=config_data,
    cli_overwrite=False,
    default_overwrite=False
)

print(f"Experiment: {runner.name}")
print(f"Workspace: {runner.base_workspace}")
print(f"Models to train: {runner.models}")

In [ ]:
# ESEGUI L'ESPERIMENTO
# ⚠️ Questo può richiedere alcuni minuti!
# ⚠️ Richiede GPU per BERT-INT

print("Starting experiment...")
print("This may take 5-10 minutes depending on your hardware.\n")

try:
    runner.run()
    print("\n✓ Experiment completed successfully!")
except Exception as e:
    print(f"\n✗ Experiment failed: {e}")
    import traceback
    traceback.print_exc()

## 4. Analizzare i Risultati

In [ ]:
import json

# Trova il file dei risultati
results_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "reduction" / "results.json"

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    print("=" * 60)
    print("RISULTATI BASELINE (30% reduction)")
    print("=" * 60)
    
    for model, metrics in results.items():
        print(f"\nModel: {model}")
        print("-" * 40)
        print(f"  Hits@1:     {metrics.get('hits@1', 'N/A'):.4f}")
        print(f"  Hits@5:     {metrics.get('hits@5', 'N/A'):.4f}")
        print(f"  Hits@10:    {metrics.get('hits@10', 'N/A'):.4f}")
        print(f"  MRR:        {metrics.get('mrr', 'N/A'):.4f}")
        print(f"  Precision:  {metrics.get('precision', 'N/A'):.4f}")
        print(f"  Recall:     {metrics.get('recall', 'N/A'):.4f}")
        print(f"  F-measure:  {metrics.get('f-measure', 'N/A'):.4f}")
else:
    print(f"✗ Results file not found: {results_file}")

## 5. Struttura dei Risultati

I risultati sono salvati in:

```
results/tutorial_baseline/
├── metadata.json              # Metadati esperimento
└── reduction_030/             # Ratio 0.3 (30%)
    └── D_W_15K_V1/            # Dataset
        ├── artifacts/         # Cache e file intermedi
        └── reduction/         # Stage reduction
            ├── results.json   # Risultati
            ├── summary.json   # Riepilogo
            └── logs/          # Log di esecuzione
```

## Metriche Spiegate

- **Hits@k**: % di volte in cui l'entità corretta è nei top-k risultati
  - Hits@1: L'entità corretta è il primo risultato
  - Hits@5: L'entità corretta è nei primi 5
  - Hits@10: L'entità corretta è nei primi 10

- **MRR** (Mean Reciprocal Rank): Media del reciproco del rank dell'entità corretta
  - Valori più alti = migliore (max 1.0)

- **Precision/Recall/F-measure**: Metriche standard di classificazione
  - Precision: % di allineamenti corretti tra quelli predetti
  - Recall: % di allineamenti trovati rispetto al totale
  - F-measure: Media armonica di precision e recall

## Prossimi Passi

Nel prossimo notebook vedremo come:
- Aggiungere data augmentation usando PLM
- Confrontare baseline vs augmented
- Analizzare il miglioramento

➡️ Continua con: **03_data_augmentation.ipynb**